In [1]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [5]:
df.isnull().sum()


AREA             0
PERIMETER        0
MAJOR_AXIS       0
MINOR_AXIS       0
ECCENTRICITY     0
EQDIASQ          0
SOLIDITY         0
CONVEX_AREA      0
EXTENT           0
ASPECT_RATIO     0
ROUNDNESS        0
COMPACTNESS      0
SHAPEFACTOR_1    0
SHAPEFACTOR_2    0
SHAPEFACTOR_3    0
SHAPEFACTOR_4    0
MeanRR           0
MeanRG           0
MeanRB           0
StdDevRR         0
StdDevRG         0
StdDevRB         0
SkewRR           0
SkewRG           0
SkewRB           0
KurtosisRR       0
KurtosisRG       0
KurtosisRB       0
EntropyRR        0
EntropyRG        0
EntropyRB        0
ALLdaub4RR       0
ALLdaub4RG       0
ALLdaub4RB       0
Class            0
dtype: int64

In [6]:
X = df.drop("Class",axis=1)
y=df["Class"]

In [9]:
from sklearn.preprocessing import StandardScaler,LabelEncoder

le = LabelEncoder()
y= le.fit_transform(y)

In [10]:
from sklearn.model_selection import train_test_split

X_train,X_test , y_train,y_test = train_test_split(
    X,y, test_size=0.2,random_state=42
    )

In [11]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### ANN


In [23]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset

In [24]:
X_train_tensor = torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train,dtype=torch.long)

X_test_tensor = torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor = torch.tensor(y_test,dtype=torch.long)

In [25]:
train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
test_dataset = TensorDataset(X_test_tensor,y_test_tensor)

In [26]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32)

In [27]:
# Model
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X.shape[1],64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,7)

        )
    def forward(self,x):
        return self.model(x)

In [28]:
model = ANN()

#loss and optim
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())


In [29]:
#Training the NNN

epochs=100
for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    for xb,yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = criteria(outputs,yb)
        loss.backward()
        optimizer.step() #param updated

        running_loss+=loss.item()
    train_loss = running_loss / len(train_loader)
    print(f"{epoch+1}/{epochs},loss = {train_loss}")

1/100,loss = 1.6878138521443242
2/100,loss = 1.1391409454138384
3/100,loss = 0.7702636161576146
4/100,loss = 0.5759128746779069
5/100,loss = 0.4603530589653098
6/100,loss = 0.405530281688856
7/100,loss = 0.3565149384996165
8/100,loss = 0.32882279481576837
9/100,loss = 0.29132669224687247
10/100,loss = 0.26800145176441775
11/100,loss = 0.2539756152292956
12/100,loss = 0.23624466359615326
13/100,loss = 0.22230977111536523
14/100,loss = 0.2150038263720015
15/100,loss = 0.20092687221324962
16/100,loss = 0.19700918524809505
17/100,loss = 0.1842486365981724
18/100,loss = 0.17166921852723413
19/100,loss = 0.16455140677483185
20/100,loss = 0.1709400817104008
21/100,loss = 0.15428191018493279
22/100,loss = 0.15468257503665012
23/100,loss = 0.14492501448030057
24/100,loss = 0.14064812951761743
25/100,loss = 0.13916870757289554
26/100,loss = 0.13108858579526778
27/100,loss = 0.13006955223238986
28/100,loss = 0.1232009071694768
29/100,loss = 0.12412423980624779
30/100,loss = 0.12763914044784463
31

In [ ]:
#Evaluate
model.eval()


total =0
correct =0

with torch.no_grad():
    for xb,yb in test_loader:
        outputs = model(xb)
        _,predicted = torch.max(outputs,1)

        correct += (predicted==yb).sum().item()
        total +=yb.size(0) #actual sample in each batch

print("Total Values",total)
print("Correct Values:" ,correct)
print("accuracy : ",correct/total *100)


Total Values 180
Correct Values: 170
accuracy :  94.44444444444444
